# Sprint 7 - Data Preparation Pipeline

This notebook demonstrates the complete data preparation pipeline for traditional machine learning models.

Pipeline:

Raw Data
→ Validation
→ RUL Generation
→ Feature Engineering
→ Feature Selection
→ Train/Validation Split
→ Feature Scaling

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent

sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

A:\AI_Engineer\ML Projects\Predictive-Maintenance-RUL


In [3]:
import pandas as pd

from src.data.loader import DataLoader
from src.data.validator import DataValidator
from src.preprocessing.rul_generator import RULGenerator
from src.preprocessing.feature_engineer import FeatureEngineer

from src.preprocessing.feature_selector import FeatureSelector
from src.preprocessing.data_splitter import DataSplitter
from src.preprocessing.feature_scaler import FeatureScaler
from src.preprocessing.preparation_pipeline import DataPreparationPipeline
from src.utils.constant import SENSOR_COLUMNS

from src.config.config import (
    TRAIN_DATA_PATH,
    TEST_DATA_PATH,
    RUL_DATA_PATH,
    INTERIM_DATA_DIR,
    FEATURES_DATA_DIR,
    PROCESSED_DATA_DIR,
    SCALERS_DIR,
)

1- LOAD DATA

In [4]:
loader = DataLoader(
    train_path=TRAIN_DATA_PATH,
    test_path=TEST_DATA_PATH,
    rul_path=RUL_DATA_PATH,
)

train_df = loader.load_train()
test_df = loader.load_test()
rul_df = loader.load_rul()

2026-08-06 17:40:58 | INFO | loader.py | Line:18 | Reading train_FD004.txt
2026-08-06 17:40:59 | INFO | loader.py | Line:21 | train_FD004.txt Loaded Successfully
2026-08-06 17:40:59 | INFO | loader.py | Line:18 | Reading test_FD004.txt
2026-08-06 17:41:01 | INFO | loader.py | Line:21 | test_FD004.txt Loaded Successfully
2026-08-06 17:41:01 | INFO | loader.py | Line:18 | Reading RUL_FD004.txt
2026-08-06 17:41:01 | INFO | loader.py | Line:21 | RUL_FD004.txt Loaded Successfully


2- VALIDATE DATA

In [5]:
validator = DataValidator(
    train_df,
    test_df,
    rul_df
)

validator.validate_all()

2026-08-06 17:41:01 | INFO | validator.py | Line:40 | Validating training dataset...
2026-08-06 17:41:01 | INFO | validator.py | Line:50 | Validating testing dataset...
2026-08-06 17:41:01 | INFO | validator.py | Line:60 | Validating RUL dataset...


{'train': {'valid': True, 'errors': [], 'warnings': []},
 'test': {'valid': True, 'errors': [], 'warnings': []},
 'rul': {'valid': True, 'errors': [], 'warnings': ['Duplicate rows found.']}}

3- Generate RUL

In [6]:
generator = RULGenerator(train_df=train_df)

train_df = generator.generate(cap=125)

train_df.head()

2026-08-06 17:41:01 | INFO | rul_generator.py | Line:82 | Generating Remaining Useful Life (RUL)...
2026-08-06 17:41:01 | INFO | rul_generator.py | Line:92 | Applying RUL cap = 125
2026-08-06 17:41:01 | INFO | rul_generator.py | Line:96 | RUL generated successfully.


,unit_number,time_in_cycles,operational_setting_1,operational_setting_2,operational_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,RUL
0,1,1,42.0049,0.8400,100.0,445.00,549.68,1343.43,1112.93,3.91,...,2387.99,8074.83,9.3335,0.02,330,2212,100.00,10.62,6.3670,125
1,1,2,20.0020,0.7002,100.0,491.19,606.07,1477.61,1237.50,9.35,...,2387.73,8046.13,9.1913,0.02,361,2324,100.00,24.37,14.6552,125
2,1,3,42.0038,0.8409,100.0,445.00,548.95,1343.12,1117.05,3.91,...,2387.97,8066.62,9.4007,0.02,329,2212,100.00,10.48,6.4213,125
3,1,4,42.0000,0.8400,100.0,445.00,548.70,1341.24,1118.03,3.91,...,2388.02,8076.05,9.3369,0.02,328,2212,100.00,10.54,6.4176,125
4,1,5,25.0063,0.6207,60.0,462.54,536.10,1255.23,1033.59,7.05,...,2028.08,7865.80,10.8366,0.02,305,1915,84.93,14.03,8.6754,125


In [7]:
INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)

train_df.to_csv(
    INTERIM_DATA_DIR / "train_with_rul.csv",
    index=False,
)

print(f"Saved train_with_rul.csv to {INTERIM_DATA_DIR}")

Saved train_with_rul.csv to A:\AI_Engineer\ML Projects\Predictive-Maintenance-RUL\artifacts\data\interim


4- Feature Engineering

In [8]:
engineer = FeatureEngineer(sensor_columns=SENSOR_COLUMNS,lags=[1,2,3],)

feature_df = engineer.transform(train_df)
feature_df.head()

2026-08-06 17:41:02 | INFO | feature_engineer.py | Line:50 | Starting Feature Engineering...
2026-08-06 17:41:02 | INFO | feature_engineer.py | Line:119 | Generating Rolling Mean features...
2026-08-06 17:41:04 | INFO | feature_engineer.py | Line:148 | Generating Rolling Std features...
2026-08-06 17:41:05 | INFO | feature_engineer.py | Line:179 | Generating Lag Features...
A:\AI_Engineer\ML Projects\Predictive-Maintenance-RUL\src\preprocessing\feature_engineer.py:189: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[feature_name] = (
2026-08-06 17:41:05 | INFO | feature_engineer.py | Line:205 | Generating Rate of Change features...
A:\AI_Engineer\ML Projects\Predictive-Maintenance-RUL\src\preprocessing\feature_engineer.py:213: PerformanceWarning: DataFrame

,unit_number,time_in_cycles,operational_setting_1,operational_setting_2,operational_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12_diff,sensor_13_diff,sensor_14_diff,sensor_15_diff,sensor_16_diff,sensor_17_diff,sensor_18_diff,sensor_19_diff,sensor_20_diff,sensor_21_diff
0,1,1,42.0049,0.8400,100.0,445.00,549.68,1343.43,1112.93,3.91,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2,20.0020,0.7002,100.0,491.19,606.07,1477.61,1237.50,9.35,...,182.81,-0.26,-28.70,-0.1422,0.0,31.0,112.0,0.00,13.75,8.2882
2,1,3,42.0038,0.8409,100.0,445.00,548.95,1343.12,1117.05,3.91,...,-182.97,0.24,20.49,0.2094,0.0,-32.0,-112.0,0.00,-13.89,-8.2339
3,1,4,42.0000,0.8400,100.0,445.00,548.70,1341.24,1118.03,3.91,...,0.18,0.05,9.43,-0.0638,0.0,-1.0,0.0,0.00,0.06,-0.0037
4,1,5,25.0063,0.6207,60.0,462.54,536.10,1255.23,1033.59,7.05,...,34.31,-359.94,-210.25,1.4997,0.0,-23.0,-297.0,-15.07,3.49,2.2578


In [9]:
print(f"Rows      : {feature_df.shape[0]}")
print(f"Columns   : {feature_df.shape[1]}")

Rows      : 61249
Columns   : 153


In [10]:
FEATURES_DATA_DIR.mkdir(parents=True, exist_ok=True)

feature_df.to_csv(
    FEATURES_DATA_DIR / "train_features.csv",
    index=False,
)

print(f"Saved train_features.csv to {FEATURES_DATA_DIR}")

Saved train_features.csv to A:\AI_Engineer\ML Projects\Predictive-Maintenance-RUL\artifacts\data\features


5- Create Pipeline

In [11]:
pipeline = DataPreparationPipeline(

    splitter=DataSplitter(
        test_size=0.2,
    ),

    selector=FeatureSelector(

        target_column="RUL",

        drop_columns=[
            "unit_number",
        ],

    ),

    scaler=FeatureScaler(),

)

In [12]:
X_train, X_val, y_train, y_val = pipeline.prepare(
    feature_df
)

2026-08-06 17:41:20 | INFO | preparation_pipeline.py | Line:37 | Starting Data Preparation Pipeline...
2026-08-06 17:41:20 | INFO | data_splitter.py | Line:35 | Starting engine-based train/validation split...
2026-08-06 17:41:20 | INFO | data_splitter.py | Line:67 | Train Engines: 199 | Validation Engines: 50
2026-08-06 17:41:20 | INFO | data_splitter.py | Line:72 | Data splitting completed successfully.
2026-08-06 17:41:20 | INFO | feature_selector.py | Line:29 | Starting Feature Selection...
2026-08-06 17:41:20 | INFO | feature_selector.py | Line:40 | Feature Selection completed successfully.
2026-08-06 17:41:20 | INFO | feature_selector.py | Line:29 | Starting Feature Selection...
2026-08-06 17:41:20 | INFO | feature_selector.py | Line:40 | Feature Selection completed successfully.
2026-08-06 17:41:20 | INFO | feature_scaler.py | Line:33 | Fitting Feature Scaler...
2026-08-06 17:41:21 | INFO | feature_scaler.py | Line:37 | Feature Scaler fitted successfully.
2026-08-06 17:41:21 | IN

In [13]:
SCALERS_DIR.mkdir(parents=True, exist_ok=True)

pipeline.scaler.save(
    SCALERS_DIR / "feature_scaler.pkl"
)

print(f"Saved feature_scaler.pkl to {SCALERS_DIR}")

2026-08-06 17:41:21 | INFO | feature_scaler.py | Line:89 | Scaler saved to A:\AI_Engineer\ML Projects\Predictive-Maintenance-RUL\artifacts\models\scalers\feature_scaler.pkl


Saved feature_scaler.pkl to A:\AI_Engineer\ML Projects\Predictive-Maintenance-RUL\artifacts\models\scalers


In [14]:
print("Training Features :", X_train.shape)
print("Validation Features :", X_val.shape,"\n")

print("Training Target :", y_train.shape)
print("Validation Target :", y_val.shape)

Training Features : (49294, 151)
Validation Features : (11955, 151) 

Training Target : (49294,)
Validation Target : (11955,)


In [15]:
X_train.describe().T[
    ["mean","std"]
].head(15).round(1)

,mean,std
time_in_cycles,-0.0,1.0
operational_setting_1,0.0,1.0
operational_setting_2,0.0,1.0
operational_setting_3,0.0,1.0
sensor_1,0.0,1.0
sensor_2,0.0,1.0
sensor_3,0.0,1.0
sensor_4,-0.0,1.0
sensor_5,0.0,1.0
sensor_6,0.0,1.0


In [16]:
display(X_train.head())

display(y_train.head())

,time_in_cycles,operational_setting_1,operational_setting_2,operational_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,sensor_6,...,sensor_12_diff,sensor_13_diff,sensor_14_diff,sensor_15_diff,sensor_16_diff,sensor_17_diff,sensor_18_diff,sensor_19_diff,sensor_20_diff,sensor_21_diff
0,-1.487344,1.216939,0.863870,0.419312,-1.053503,-0.794792,-0.698775,-0.742599,-1.136434,-1.080525,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,-1.476237,-0.271311,0.413981,0.419312,0.692954,0.714630,0.564431,0.301134,0.364435,0.371788,...,0.930044,-0.002341,-0.243403,-0.133916,-0.000216,0.785921,0.543754,-0.000892,0.974918,0.979361
2,-1.465130,1.216864,0.866766,0.419312,-1.053503,-0.814332,-0.701693,-0.708079,-1.136434,-1.082361,...,-0.929700,0.000424,0.169890,0.198881,-0.000216,-0.811806,-0.544481,-0.000892,-0.983445,-0.971549
3,-1.454023,1.216607,0.863870,0.419312,-1.053503,-0.821024,-0.719392,-0.699867,-1.136434,-1.080525,...,0.001494,-0.000627,0.076964,-0.059709,-0.000216,-0.025623,-0.000363,-0.000892,0.004947,0.000263
4,-1.442917,0.067174,0.158143,-2.384860,-0.390311,-1.158295,-1.529112,-1.407363,-0.270124,-0.474629,...,0.175022,-1.991466,-1.768783,1.420180,-0.000216,-0.583560,-1.443246,-1.990685,0.247971,0.267298


0    125
1    125
2    125
3    125
4    125
Name: RUL, dtype: int64

6- Save Prepared Data

In [17]:
train_prepared = X_train.copy()
train_prepared["RUL"] = y_train.values

validation_prepared = X_val.copy()
validation_prepared["RUL"] = y_val.values

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

train_prepared.to_csv(
    PROCESSED_DATA_DIR / "train_prepared.csv",
    index=False,
)

validation_prepared.to_csv(
    PROCESSED_DATA_DIR / "validation_prepared.csv",
    index=False,
)

print(f"Saved train_prepared.csv and validation_prepared.csv to {PROCESSED_DATA_DIR}")

Saved train_prepared.csv and validation_prepared.csv to A:\AI_Engineer\ML Projects\Predictive-Maintenance-RUL\artifacts\data\processed


## Engineering Decisions

### Data Leakage Prevention

- Target generated before splitting.
- Feature Engineering performed independently for each engine.
- Train/Validation split performed by engine IDs.
- Scaler fitted only on the training data.

---

### Feature Selection

Removed:

- unit_number

Target:

- RUL

---

### Scaling

StandardScaler

Fit:

Training only

Transform:

Validation only

---

### Missing Values

NaN values generated by

- Rolling Mean
- Rolling Std
- Lag Features

These will be handled in the next sprint.

---

### Output

The pipeline now returns

X_train
X_validation
y_train
y_validation

ready for model training.